# SpectraLite — Research Lab (`works.ipynb`)

**Single notebook for the whole project.** Open this file via:

`File → Open notebook → GitHub → SpectraLite/notebooks/works.ipynb`

Do **not** double-click notebooks in the left Files panel (raw text mode).

| Stage | Status in this file | Depends on |
|-------|---------------------|------------|
| 0 Bootstrap + env | **Runnable now** | Fresh Colab GPU runtime |
| Phase 0 smoke test | **Done (keep outputs)** | Stage 0 |
| Phase 1 baseline harness | **Runnable now** | Phase 0 PASS on GPU |
| Phase 2 vanilla SVD | Placeholder | Phase 1 metrics schema |
| Phase 3 ASVD / SVD-LLM | Placeholder | Phase 2 |
| Phase 4 SpectraLite ranks | Placeholder | Phase 3 |
| Phase 5 stability | Placeholder | Phase 4 |
| Phase 6 latency engineering | Placeholder | Phase 5 |
| Phase 7 ablations | Placeholder | Phase 6 |
| Phase 8 downstream eval | Placeholder | Chosen operating point |

Artifacts under `results/` (`phase_status.json`, summaries, CSVs) are **git-tracked** so a new Colab can skip completed phases after `git pull`.

When a stage is ready, we fill its section here. If it needs prior outputs, run the previous section, save/push this notebook, and tell Cursor to continue.


---
# Stage 0 — Colab Bootstrap

**Run once per new Colab runtime** (A100 recommended).

- Clones / pulls the full GitHub repo
- Installs `requirements-colab.txt` (does **not** reinstall torch)
- Verifies CUDA


### 0.1 Runtime check

In [1]:
import sys
print("In Colab:", "google.colab" in sys.modules)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        raise SystemExit(
            "Select Runtime → Change runtime type → GPU (A100), "
            "then Disconnect and delete runtime if CUDA stays False."
        )
except ImportError as exc:
    raise SystemExit(f"torch missing: {exc}") from exc


In Colab: True
Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


### 0.2 Clone or pull repo

In [2]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/PrabinDevkota/Execution.git"
CLONE_ROOT = Path("/content/Execution")
PACKAGE_ROOT = CLONE_ROOT / "SpectraLite"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not PACKAGE_ROOT.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(CLONE_ROOT)])
    else:
        subprocess.check_call(["git", "-C", str(CLONE_ROOT), "pull", "--ff-only"])
    if str(PACKAGE_ROOT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_ROOT))
    %cd {PACKAGE_ROOT}
else:
    # Local: expect cwd inside SpectraLite or notebooks/
    cwd = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for candidate in [cwd, cwd.parent]:
        if (candidate / "spectralite").is_dir():
            PACKAGE_ROOT = candidate
            break
    if PACKAGE_ROOT is None:
        raise FileNotFoundError("Run from SpectraLite repo locally.")
    if str(PACKAGE_ROOT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_ROOT))

print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("spectralite OK =", (PACKAGE_ROOT / "spectralite").is_dir())


/content/Execution/SpectraLite
PACKAGE_ROOT = /content/Execution/SpectraLite
spectralite OK = True


### 0.3 Install dependencies + CUDA verify

In [3]:
from spectralite.colab_setup import ensure_environment, in_colab

# On Colab: install requirements-colab.txt (no torch). Local: requirements.txt.
repo_root = ensure_environment(
    pull=False,
    install=True,
    require_cuda_on_colab=True,
)
print("Stage 0 complete. repo_root =", repo_root)


SpectraLite environment ready
  repo_root      : /content/Execution/SpectraLite
  requirements   : /content/Execution/SpectraLite/requirements-colab.txt
  torch          : 2.11.0+cu128
  cuda_available : True
  gpu            : NVIDIA A100-SXM4-40GB
Stage 0 complete. repo_root = /content/Execution/SpectraLite


---
# Session Restore (new Colab / continuing work)

Run this **after Stage 0** on every new runtime.

- Reads `results/phase_status.json` from git
- Skips recomputing finished phases
- Reloads the HF model only if a later phase still needs weights  
  (weights are **not** stored in git — too large)


In [ ]:
from spectralite.session import restore_session
from spectralite.artifacts import print_progress_dashboard

# Set True only if you intentionally want to redo finished measurements
FORCE_RERUN_PHASE0 = False
FORCE_RERUN_PHASE1 = False

session = restore_session(load_model_if_needed=True)
cfg = session["cfg"]
status = session["status"]

# Prefer restored in-memory model when present
if session["model"] is not None:
    model = session["model"]
    tokenizer = session["tokenizer"]

SKIP_PHASE0 = session["skip_phase0"] and not FORCE_RERUN_PHASE0
SKIP_PHASE1 = session["skip_phase1"] and not FORCE_RERUN_PHASE1

print("SKIP_PHASE0 =", SKIP_PHASE0)
print("SKIP_PHASE1 =", SKIP_PHASE1)
if SKIP_PHASE0:
    print("Phase 0 artifacts already in git — you may skip Phase 0 cells.")
if SKIP_PHASE1:
    print("Phase 1 artifacts already in git — you may skip Phase 1 measurement cells.")


---
# Phase 0 — Environment & Model Smoke Test

**Goal:** Load `facebook/opt-125m` in FP16 on CUDA, inventory every `nn.Linear`, run one generation, report GPU memory.

**No compression / SVD / whitening here.**


In [ ]:
if globals().get("SKIP_PHASE0", False):
    print("Phase 0 already recorded in results/ — skipping recompute.")
    print("Jump to Phase 1, or set FORCE_RERUN_PHASE0=True in Session Restore.")
else:
    print("Run Phase 0 cells below, then commit results/ to git.")


### P0.1 Config, seed, env report

In [4]:
from spectralite import __version__, default_config, set_seed, gpu
from spectralite.system import print_environment_report
from spectralite.utils import get_logger, print_section

cfg = default_config()
cfg.ensure_directories()
set_seed(cfg.seed)
logger = get_logger("spectralite.works")

print_section("SpectraLite works.ipynb — Phase 0")
print(f"  package     : {__version__}")
print(f"  model       : {cfg.model_name}")
print(f"  seed        : {cfg.seed}")
print(f"  dtype       : {cfg.dtype}")
print(f"  device_map  : {cfg.device_map}")

env_info = print_environment_report()
environment_ready = env_info["pytorch_version"] != "unavailable"
gpu_ready = bool(env_info["cuda_available"])
assert gpu_ready, "GPU required for SpectraLite Phase 0+"



 SpectraLite works.ipynb — Phase 0
  package     : 0.1.0-phase0
  model       : facebook/opt-125m
  seed        : 42
  dtype       : float16
  device_map  : auto

 Environment Verification
  Python Version               3.12.13
  PyTorch Version              2.11.0+cu128
  CUDA Version                 12.8
  cuDNN Version                91900
  GPU Name                     NVIDIA A100-SXM4-40GB
  GPU Memory                   39.49 GB
  Torch CUDA Available         True
  Device                       cuda
  GPU Count                    1
  Platform                     Linux-6.6.122+-x86_64-with-glibc2.35


### P0.2 GPU memory before load

In [5]:
mem_before = gpu.snapshot(label="Memory before loading")
gpu.print_memory_snapshot(mem_before)



 Memory before loading
  Allocated                    0 B
  Reserved                     0 B
  Free (approx)                39.49 GB
  Total                        39.49 GB


{'label': 'Memory before loading',
 'cuda_available': True,
 'device_index': 0,
 'allocated_bytes': 0,
 'reserved_bytes': 0,
 'free_bytes': 42405855232,
 'total_bytes': 42405855232,
 'allocated': '0 B',
 'reserved': '0 B',
 'free': '39.49 GB',
 'total': '39.49 GB'}

### P0.3 Load model + tokenizer

In [6]:
from spectralite.model_loader import load_model_and_tokenizer, print_model_load_summary

model, tokenizer = load_model_and_tokenizer(config=cfg)
load_summary = print_model_load_summary(model, model_name=cfg.model_name)

mem_after_load = gpu.snapshot(label="Memory after loading")
gpu.print_memory_snapshot(mem_after_load)

model_loaded = model is not None and tokenizer is not None
logger.info("Loaded %s on %s", cfg.model_name, load_summary["device"])


2026-07-12 15:53:59 | INFO     | spectralite.model_loader | Loading tokenizer: facebook/opt-125m


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

2026-07-12 15:54:02 | INFO     | spectralite.model_loader | Loading model: facebook/opt-125m | dtype=torch.float16 | device_map=auto


pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


 Model Load Summary
  Model Name                   facebook/opt-125m
  Architecture                 OPTForCausalLM
  Total Parameters             125.24M (125,239,296)
  Trainable Parameters         125.24M (125,239,296)
  Model Size (params)          238.88 MB
  Tensor Data Type             float16
  Current Device               cuda:0

 Memory after loading
  Allocated                    241.50 MB
  Reserved                     358.00 MB
  Free (approx)                39.14 GB
  Total                        39.49 GB
2026-07-12 15:54:09 | INFO     | spectralite.works | Loaded facebook/opt-125m on cuda:0


### P0.4 Model analysis — every `nn.Linear`

In [7]:
from spectralite.model_analysis import print_full_model_analysis

analysis = print_full_model_analysis(model, model_name=cfg.model_name)
print(f"\nBlocks (first 3): {analysis['block_names'][:3]}")
print(f"Linear layers: {analysis['num_linear_layers']}")

# Keep for later phases (SVD targets)
LINEAR_LAYERS = analysis["linear_layers"]



 Model Information
  Model Name                   facebook/opt-125m
  Architecture                 OPTForCausalLM
  Transformer Blocks           12
  Attention Linear Layers      48
  MLP Linear Layers            24
  Other Linear Layers          1
  Total nn.Linear Layers       73
  Total Parameters             125.24M (125,239,296)
  Trainable Parameters         125.24M (125,239,296)
  Model Size / Memory Footprint 238.88 MB
  Tensor Data Type             float16
  Current Device               cuda:0

 nn.Linear Inventory (73 layers)
  #    Layer Name                                                  In    Out Weight Shape    
  -------------------------------------------------------------------------------------------
  0    model.decoder.layers.0.self_attn.k_proj                    768    768 (768, 768)      
  1    model.decoder.layers.0.self_attn.v_proj                    768    768 (768, 768)      
  2    model.decoder.layers.0.self_attn.q_proj                    768    768 (768

### P0.5 Test inference

In [8]:
from spectralite.model_loader import generate_text

generated = generate_text(
    model,
    tokenizer,
    cfg.smoke_prompt,
    max_new_tokens=cfg.max_new_tokens,
    do_sample=False,
)

print_section("Test Inference")
print("Prompt:", cfg.smoke_prompt)
print("-" * 60)
print(generated)
print("-" * 60)

inference_ok = isinstance(generated, str) and len(generated.strip()) > 0



 Test Inference
Prompt: Explain Singular Value Decomposition in one sentence.
------------------------------------------------------------
Explain Singular Value Decomposition in one sentence.

I’m not sure what you mean by “explain”.

I’m not sure what you mean by “explain”.

I’m not sure what you mean by
------------------------------------------------------------


### P0.6 GPU memory after inference + checklist

In [9]:
from spectralite.utils import print_checklist

mem_after_infer = gpu.snapshot(label="Memory after inference")
gpu.print_memory_snapshot(mem_after_infer)
gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])

phase0_complete = all([environment_ready, gpu_ready, model_loaded, inference_ok])
print_checklist(
    [
        ("Environment Ready", environment_ready),
        ("GPU Ready", gpu_ready),
        ("Model Loaded Successfully", model_loaded),
        ("Inference Successful", inference_ok),
        ("Phase 0 Complete", phase0_complete),
    ]
)

print_section("Phase 0 Outcome")
for line in [
    "Environment Ready",
    "GPU Ready",
    "Model Loaded Successfully",
    "Inference Successful",
    "Phase 0 Complete",
]:
    print(" ", line)

assert phase0_complete, "Phase 0 incomplete — fix GPU/load/inference before Phase 1"


# --- Persist Phase 0 to git-tracked results/ ---
from spectralite.artifacts import save_phase0_artifacts, print_git_save_instructions

if phase0_complete:
    save_phase0_artifacts(
        env_info=env_info,
        load_summary=load_summary,
        analysis=analysis,
        inference_ok=inference_ok,
        config=cfg,
    )
    print_git_save_instructions()



 Memory after inference
  Allocated                    250.62 MB
  Reserved                     362.00 MB
  Free (approx)                39.14 GB
  Total                        39.49 GB

 GPU Memory Timeline
  Memory before loading        alloc=0 B          reserved=0 B          free=39.49 GB
  Memory after loading         alloc=241.50 MB    reserved=358.00 MB    free=39.14 GB
  Memory after inference       alloc=250.62 MB    reserved=362.00 MB    free=39.14 GB

 Phase 0 Status
  [PASS] Environment Ready
  [PASS] GPU Ready
  [PASS] Model Loaded Successfully
  [PASS] Inference Successful
  [PASS] Phase 0 Complete

 Phase 0 Outcome
  Environment Ready
  GPU Ready
  Model Loaded Successfully
  Inference Successful
  Phase 0 Complete


---
# Phase 1 — Baseline Profiling Harness

**Status:** Runnable (dense baseline for `facebook/opt-125m`).

**Depends on:** Stage 0 + model available (from Phase 0 or Session Restore).

If `SKIP_PHASE1` is True (artifacts already in git), the measurement cell loads cached
metrics instead of recomputing.

**Writes (commit these):**
- `results/phase1_dense_baselines.csv`
- `results/phase1_summary.json`
- `results/phase_status.json`


### P1.0 Ensure model is available

In [ ]:
from spectralite import default_config, set_seed
from spectralite.model_loader import load_model_and_tokenizer
from spectralite.utils import print_section

if "cfg" not in globals():
    cfg = default_config()
    cfg.ensure_directories()
    set_seed(cfg.seed)

if "SKIP_PHASE1" not in globals():
    from spectralite.artifacts import should_skip_phase
    SKIP_PHASE1 = should_skip_phase("1", config=cfg)

if "model" not in globals() or model is None or "tokenizer" not in globals() or tokenizer is None:
    print_section("Loading model for Phase 1")
    model, tokenizer = load_model_and_tokenizer(config=cfg)
else:
    print_section("Using in-memory model")
    print("device:", next(model.parameters()).device)


### P1.1 Run dense baseline (or load cached results)

In [ ]:
from pathlib import Path
from spectralite.benchmark import run_phase1_dense_baseline
from spectralite.artifacts import read_json, is_phase_complete, print_git_save_instructions
from spectralite.results_io import load_results
from spectralite.utils import print_checklist, print_section

FORCE_RERUN_PHASE1 = globals().get("FORCE_RERUN_PHASE1", False)

if SKIP_PHASE1 and not FORCE_RERUN_PHASE1 and is_phase_complete("1", cfg):
    print_section("Phase 1 already complete — loading cached artifacts")
    phase1_summary = read_json("phase1_summary.json", cfg)
    row = phase1_summary.get("row", {})
    phase1 = {"row": row, "csv_path": phase1_summary.get("csv_path"), "cached": True}
    print(row)
else:
    phase1 = run_phase1_dense_baseline(
        model,
        tokenizer,
        config=cfg,
        prompt_len=cfg.latency_prompt_len,
        gen_tokens=cfg.latency_gen_tokens,
        latency_warmup=cfg.latency_warmup,
        latency_reps_prefill=cfg.latency_reps_prefill,
        latency_reps_decode=cfg.latency_reps_decode,
        ppl_seq_len=cfg.ppl_seq_len,
        ppl_max_tokens=cfg.ppl_max_tokens,
        run_calflops=True,
        run_ppl=True,
        csv_name="phase1_dense_baselines.csv",
    )
    row = phase1["row"]

phase1_ok = row.get("empirical_flops_fwd") is not None and row.get("prefill_ms_mean") is not None

print_checklist(
    [
        ("FLOPs available", row.get("empirical_flops_fwd") is not None),
        ("Prefill latency available", row.get("prefill_ms_mean") is not None),
        ("Decode latency available", row.get("decode_ms_per_token_mean") is not None),
        ("Phase 1 Complete", phase1_ok and is_phase_complete("1", cfg)),
    ]
)
print_section("Phase 1 Outcome")
print("  Prefill ms:", row.get("prefill_ms_mean"))
print("  Decode ms/token:", row.get("decode_ms_per_token_mean"))
print("  WT2 PPL:", row.get("ppl_wikitext2"))
print_git_save_instructions()
assert phase1_ok, "Phase 1 incomplete"


### P1.2 Inspect results table

In [ ]:
from pathlib import Path
from spectralite.results_io import load_results
from spectralite.artifacts import print_progress_dashboard

csv_path = Path(cfg.results_dir) / "phase1_dense_baselines.csv"
df = load_results(csv_path)
print("rows:", len(df))
if len(df):
    cols = [c for c in [
        "phase", "method", "model_name", "param_count",
        "empirical_flops_fwd", "prefill_ms_mean", "decode_ms_per_token_mean",
        "throughput_tokens_per_s", "ppl_wikitext2", "ppl_ptb", "ppl_c4",
    ] if c in df.columns]
    print(df[cols].tail().to_string(index=False))
print_progress_dashboard(cfg)


### P1.3 Optional lm-eval smoke (skip by default)

In [ ]:
RUN_LM_EVAL_SMOKE = False
if not RUN_LM_EVAL_SMOKE:
    print("lm-eval smoke skipped.")
else:
    print("Enable lm_eval CLI here when needed (Phase 8).")


---
# Phase 2 — Vanilla Truncated SVD

**Status:** Placeholder.

**Depends on:** Phase 1 harness + `LINEAR_LAYERS` inventory from Phase 0.

**Will add:** Eckart–Young truncation, `LowRankLinear`, uniform rank ratios, PPL/FLOP/latency curves.


In [ ]:
# PHASE 2 PLACEHOLDER
raise NotImplementedError("Phase 2 vanilla SVD not implemented yet.")


---
# Phase 3 — Activation-Aware Baselines (ASVD / SVD-LLM)

**Status:** Placeholder.

**Depends on:** Phase 2 comparison tables + calibration data pipeline from Phase 1.


In [ ]:
# PHASE 3 PLACEHOLDER
raise NotImplementedError("Phase 3 activation-aware baselines not implemented yet.")


---
# Phase 4 — SpectraLite Spectral Rank Allocation (novelty core)

**Status:** Placeholder.

**Depends on:** Phase 3 whitening utilities + Phase 1 FLOP budget accounting.


In [ ]:
# PHASE 4 PLACEHOLDER
raise NotImplementedError("Phase 4 SpectraLite rank allocation not implemented yet.")


---
# Phase 5 — Stability Safeguards

**Status:** Placeholder.

**Depends on:** Phase 4 allocator.

**Will add:** Ledoit–Wolf shrinkage, condition-number gating, few-sample robustness.


In [ ]:
# PHASE 5 PLACEHOLDER
raise NotImplementedError("Phase 5 stability safeguards not implemented yet.")


---
# Phase 6 — Latency Gate + Real Speedup Engineering

**Status:** Placeholder.

**Depends on:** Phase 5 compressed checkpoints + Phase 1 latency harness.

**Will add:** `r < κ_speed · mn/(m+n)` gate, factor fusion, packed MLP, CUDA-graph decode.


In [ ]:
# PHASE 6 PLACEHOLDER
raise NotImplementedError("Phase 6 latency engineering not implemented yet.")


---
# Phase 7 — Ablations

**Status:** Placeholder.

**Depends on:** Phases 4–6 complete enough to toggle components.


In [ ]:
# PHASE 7 PLACEHOLDER
raise NotImplementedError("Phase 7 ablations not implemented yet.")


---
# Phase 8 — Downstream Evaluation

**Status:** Placeholder.

**Depends on:** Chosen compression operating points from Phases 4–6.

**Will add:** lm-eval zero-shot suite; optional encoder GLUE bridge.


In [ ]:
# PHASE 8 PLACEHOLDER
raise NotImplementedError("Phase 8 downstream eval not implemented yet.")


---
# Commit artifacts from Colab (after each phase)

This keeps progress in git so the next runtime only runs unfinished phases.


In [ ]:
# Optional one-cell git commit of results (requires push credentials / public write access)
COMMIT_FROM_COLAB = False  # set True when ready to push results
COMMIT_MESSAGE = "Record SpectraLite phase results from Colab"

if not COMMIT_FROM_COLAB:
    print("Set COMMIT_FROM_COLAB = True to commit results/ from Colab.")
    print("Or download results/ and commit locally in Cursor.")
else:
    import subprocess
    from pathlib import Path
    root = Path("/content/Execution")
    subprocess.check_call(["git", "-C", str(root), "add",
                           "SpectraLite/results/*.json",
                           "SpectraLite/results/*.csv",
                           "SpectraLite/notebooks/works.ipynb"], shell=False)
    # git add with globs may need running via bash on some systems:
    subprocess.check_call(
        "git add SpectraLite/results/*.json SpectraLite/results/*.csv SpectraLite/notebooks/works.ipynb",
        cwd=str(root),
        shell=True,
    )
    subprocess.check_call(["git", "-C", str(root), "status"])
    subprocess.call(["git", "-C", str(root), "commit", "-m", COMMIT_MESSAGE])
    print("Now: git push (authenticate if private/public write).")


---
# How to continue with Cursor

1. Open **this** notebook from GitHub (not Files double-click).
2. Run **Stage 0** then **Phase 0** only.
3. `File → Save a copy in GitHub`.
4. Message Cursor: which phase to implement next (e.g. Phase 1).
5. `git pull` in Stage 0.2 and run the new section.

Older notebooks (`Colab_Bootstrap.ipynb`, `Phase0_Setup.ipynb`) are legacy; **`works.ipynb` is the main lab file.**
